### 0. Read data & Import lib 

In [ ]:
import duckdb

clean_events = duckdb.sql('SELECT * FROM "../dataset/processed/clean_events.parquet";')
clean_inventory_items = duckdb.sql('SELECT * FROM "../dataset/processed/clean_inventory_items.parquet";')
clean_orders =duckdb.sql("SELECT * FROM read_parquet('../dataset/processed/clean_orders.parquet')")
clean_order_items = duckdb.sql('SELECT * FROM "../dataset/processed/clean_order_items.parquet"')
clean_users = duckdb.sql('SELECT * FROM "../dataset/processed/clean_users.parquet"')

### Denormalization: One Big Table (OBT) Architecture

#### 2.1 Frontend Traffic OBT (`Frontend_Traffic_OBT`)

***Data Modeling: Semantic Resolution & Attribution Foundation***
To create a flattened, analytical-ready table for web traffic analysis, we denormalized the `clean_events` and `clean_users` tables using a `LEFT JOIN`. This preserves our entire top-of-funnel traffic (including anonymous `user_id = 0` sessions) while enriching authenticated sessions with demographic metadata.

**Engineering Highlights:**
- **Semantic Resolution:** We explicitly resolved overlapping column names to prevent ambiguous `_x`/`_y` suffixes. 
- **Attribution Enablement:** By distinguishing between `user_traffic_source` (acquisition channel) and `event_traffic_source` (session channel), as well as `user_home_state` vs. `event_ip_state`, this OBT is natively structured to support multi-touch attribution modeling and geographic anomaly detection (e.g., fraud/traveling users).

In [ ]:
Frontend_Traffic_OBT =duckdb.sql("""
SELECT
    e.id AS event_id,
    e.user_id AS user_id,
    session_id,
    u.created_at AS user_created_at,
    e.created_at AS event_created_at,
    event_type,
    sequence_number,
    browser,
    u.traffic_source AS user_traffic_source,
    e.traffic_source AS event_traffic_source,
    age,
    gender,
    u.state AS user_home_state,
    e.state AS event_ip_state,
    u.city AS user_home_city,
    e.city AS event_ip_city,
    u.postal_code AS user_postal_code,
    latitude,
    longitude,
    uri
FROM clean_events AS e
LEFT JOIN clean_users AS u ON e.user_id = u.id
ORDER BY user_id
""")
Frontend_Traffic_OBT.to_parquet('../dataset/processed/Frontend_Traffic_OBT.parquet')

#### 2.2 Backend Sales OBT (`Backend_Sales_OBT`)

***Data Modeling: Unit Economics & Supply Chain Visibility***
This table serves as the central transactional engine of our analytical data warehouse. By denormalizing inventory, order items, orders, and user demographics into a single wide table, we establish the foundation for deep profitability and logistics analysis.

**Engineering Highlights:**
1. **Inventory-First Grain (The Left Join Strategy):** By anchoring the CTE on `clean_inventory_items` and using a `LEFT JOIN` to `order_items`, this OBT doesn't just track sales—it tracks **unsold inventory**. This architectural decision enables crucial downstream supply chain metrics, such as Sell-Through Rates, Inventory Aging, and Dead Stock identification.
2. **Granular Unit Economics:** Maintaining item-level granularity allows for precise financial modeling (e.g., Gross Margin = `sale_price - cost`) at the SKU and Category level, rather than relying on opaque aggregated order totals.
3. **Lifecycle Disambiguation:** Explicitly aliasing chronological timestamps and statuses (`order_created_at` vs. `items_created_at`, `orders_status` vs. `items_status`) prevents analytical confusion, allowing us to track the exact fulfillment micro-lifecycle of an individual item distinct from its parent order.

In [ ]:
Backend_Sales_OBT= duckdb.sql("""
WITH t1 AS(
    SELECT
        ii.id AS inventory_item_id,
        product_id,
        it.id AS order_items_id,
        product_distribution_center_id,
        ii.created_at AS inventory_items_created_at,
        sold_at,
        cost,
        product_category,
        product_name,
        product_brand,
        product_retail_price,
        product_department,
        product_sku,
        order_id,
        user_id,
        status AS items_status,
        it.created_at AS items_created_at,
        shipped_at AS items_shipped_at,
        delivered_at AS items_delivered_at,
        returned_at AS items_returned_at,
        sale_price
    FROM
        clean_inventory_items AS ii
    LEFT JOIN
        clean_order_items AS it ON ii.id = it.inventory_item_id
    )
SELECT
    t1.order_id AS order_id,
    order_items_id,
    inventory_item_id,
    product_id,
    t1.user_id AS user_id,
    product_distribution_center_id,
    o.created_at AS order_created_at,
    returned_at AS order_returned_at,
    shipped_at AS order_shipped_at,
    delivered_at AS order_delivered_at,
    items_created_at,
    items_returned_at,
    items_shipped_at,
    items_delivered_at,
    inventory_items_created_at,
    sold_at,
    u.created_at AS user_created_at,
    items_status,
    status AS orders_status,
    cost,
    product_retail_price,
    num_of_item,
    sale_price,
    age,
    u.gender,
    product_brand,
    product_department,
    product_category,
    product_name,
    traffic_source,
    state,
    postal_code,
    country,
    city,
    latitude,
    longitude,
    product_sku
FROM
    t1
LEFT JOIN
    clean_orders AS o ON t1.order_id = o.order_id
LEFT JOIN
    clean_users AS u ON t1.user_id = u.id
""")
Backend_Sales_OBT.to_parquet('../dataset/processed/Backend_Sales_OBT.parquet')